# SPORTLIS U-Net Extended — Analysis & Basin Time Series

Analisi dei risultati del modello U-Net esteso (1985–2024, dominio 484×698 @ 3km):
1. XAI permutation importance barplot
2. Performance per bacino (Cascades, Sierra Nevada, Rockies)
3. Time series SWE predetto vs SPORTLIS per bacino
4. Proiezione 2025–2026

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn.functional as F
import netCDF4 as nc
import warnings, gc
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import uniform_filter

warnings.filterwarnings('ignore')

# ── Auto-detect environment ───────────────────────────────────────────────────
_ON_NCAR  = Path('/glade/u/home/cionni').exists()
_ON_SMCE  = (not _ON_NCAR) and Path('/home/jovyan/efs').exists()

if _ON_NCAR:
    NCAR_HOME         = Path('/glade/u/home/cionni')
    NCAR_SCRATCH      = NCAR_HOME / 'derecho_scratch'
    PROJECT_DIR       = NCAR_HOME / 'unet_sportlis'
    SPORTLIS_DATA_DIR = NCAR_SCRATCH / 'sportlis_swe'
    ZARR_DIR          = NCAR_SCRATCH / 'zarr_extended'
    MEMMAP_DIR        = NCAR_SCRATCH / 'memmap_extended'
    OUTPUT_DIR        = PROJECT_DIR  / 'output_extended'
    AUX_DIR           = PROJECT_DIR  / 'auxiliary'
else:
    PROJECT_ROOT      = Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project')
    SPORTLIS_DATA_DIR = PROJECT_ROOT / 'data/inputs'
    ZARR_DIR          = PROJECT_ROOT / 'prepared_pp_narr_sportlis_extended'
    MEMMAP_DIR        = PROJECT_ROOT / 'memmap_extended'
    OUTPUT_DIR        = Path('sportlis_unet_extended_loyo_outputs')
    AUX_DIR           = PROJECT_ROOT / 'auxiliary'

FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Dominio ───────────────────────────────────────────────────────────────────
EXT_LAT_MIN, EXT_LAT_MAX = 35.005, 49.495
EXT_LON_MIN, EXT_LON_MAX = -124.93, -104.01
EXT_H, EXT_W = 484, 698
lat_1d = np.linspace(EXT_LAT_MIN, EXT_LAT_MAX, EXT_H)
lon_1d = np.linspace(EXT_LON_MIN, EXT_LON_MAX, EXT_W)

# ── Canali ────────────────────────────────────────────────────────────────────
INPUT_VARS         = ['precip_7d','precip_30d','precip_60d','precip_wytd',
                      'tair_30d_mean','tair_30d_max','degree_day_30d']
TOPO_VARS          = ['elevation','slope','aspect_sin','aspect_cos',
                      'tpi_short','tpi_long','canopy_fraction']
CHANNEL_NAMES      = INPUT_VARS + ['lat_norm','lon_norm'] + TOPO_VARS
CHANNEL_GROUP      = ['temporal']*7 + ['coord','coord'] + ['topo']*7
TARGET_VAR         = 'swe_target_filled'
MASK_VAR           = 'swe_mask'
N_IN_CHANNELS      = 16
LOYO_STEP          = 5
YEARS_AVAILABLE    = list(range(1985, 2025))
LOYO_FOLDS         = [{'test': y, 'val': y+1 if y+1 in YEARS_AVAILABLE else y-1}
                       for y in YEARS_AVAILABLE[::LOYO_STEP]]
CHECKPOINT_TEMPLATE = 'best_unet_extended_LOYO_test{test}.pt'
DROPOUT_P           = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Folds: {len(LOYO_FOLDS)}')

In [ ]:
# ── Definizione bacini (coordinate geografiche da letteratura) ─────────────────
# Riferimenti: Hamlet & Lettenmaier 2007, Mote et al. 2018, Margulis et al. 2016
# Dominio: lat [35.005, 49.495], lon [-124.93, -104.01]

BASINS = {
    'Pacific_NW_Cascades': {
        'lat': (46.0, 49.5), 'lon': (-124.5, -120.0),
        'color': '#1f77b4', 'label': 'Pacific NW Cascades'
    },
    'Oregon_Cascades': {
        'lat': (42.0, 46.0), 'lon': (-123.0, -120.0),
        'color': '#17becf', 'label': 'Oregon Cascades'
    },
    'Sierra_Nevada': {
        'lat': (36.0, 40.5), 'lon': (-121.5, -117.5),
        'color': '#ff7f0e', 'label': 'Sierra Nevada'
    },
    'Northern_Rockies': {
        'lat': (44.0, 49.5), 'lon': (-117.0, -110.0),
        'color': '#2ca02c', 'label': 'Northern Rockies'
    },
    'Southern_Rockies': {
        'lat': (36.5, 41.5), 'lon': (-109.0, -104.5),
        'color': '#d62728', 'label': 'Southern Rockies (CO)'
    },
}

def make_basin_mask(lat_1d, lon_1d, lat_range, lon_range):
    """Maschera booleana (H, W) per un bacino rettangolare."""
    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    return ((lat2d >= lat_range[0]) & (lat2d <= lat_range[1]) &
            (lon2d >= lon_range[0]) & (lon2d <= lon_range[1]))

basin_masks = {}
for name, info in BASINS.items():
    mask = make_basin_mask(lat_1d, lon_1d, info['lat'], info['lon'])
    basin_masks[name] = mask
    print(f"{info['label']:30s}: {mask.sum():6d} pixel ({100*mask.sum()/(EXT_H*EXT_W):.1f}% del dominio)")

# ── Mappa dei bacini ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
basin_map = np.zeros((EXT_H, EXT_W))
for i, (name, mask) in enumerate(basin_masks.items(), 1):
    basin_map[mask] = i

colors_list = ['white'] + [BASINS[n]['color'] for n in basin_masks]
from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors_list)
ax.imshow(basin_map, origin='upper', cmap=cmap,
          extent=[EXT_LON_MIN, EXT_LON_MAX, EXT_LAT_MIN, EXT_LAT_MAX],
          aspect='auto', alpha=0.7, vmin=0, vmax=len(BASINS))
patches = [mpatches.Patch(color=info['color'], label=info['label'])
           for info in BASINS.values()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Bacini di analisi nel dominio SPORTLIS esteso')
plt.tight_layout()
plt.savefig(FIG_DIR / 'basin_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: basin_map.png')

In [ ]:
# ── XAI Barplot ───────────────────────────────────────────────────────────────
xai_npz = OUTPUT_DIR / 'xai_extended_importance.npz'
xai_csv = OUTPUT_DIR / 'xai_extended_importance.csv'

if not xai_npz.exists():
    print(f'File non trovato: {xai_npz}')
else:
    xai = np.load(str(xai_npz), allow_pickle=True)
    imp_matrix   = xai['imp_matrix']        # (n_folds, n_channels)
    channel_names = list(xai['channel_names'])
    channel_group = list(xai['channel_group'])
    mae_baseline  = xai['mae_baseline']

    imp_mean = np.nanmean(imp_matrix, axis=0)
    imp_std  = np.nanstd(imp_matrix,  axis=0)
    order    = np.argsort(-imp_mean)

    gc_color = {'temporal': '#d32f2f', 'topo': '#388e3c', 'coord': '#616161'}
    names_s  = [channel_names[i] for i in order]
    grps_s   = [channel_group[i] for i in order]
    mean_s   = imp_mean[order]
    std_s    = imp_std[order]
    colors   = [gc_color.get(g, 'grey') for g in grps_s]

    fig, ax = plt.subplots(figsize=(10, 7))
    yp = np.arange(len(names_s))
    ax.barh(yp, mean_s, xerr=std_s, color=colors, edgecolor='k',
            alpha=0.85, capsize=3, height=0.7)
    ax.set_yticks(yp); ax.set_yticklabels(names_s, fontsize=10)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Importanza permutazione (ΔMAE mm)', fontsize=11)
    ax.set_title('XAI Permutation Importance — U-Net SWE Extended (1985–2024)', fontsize=12)
    patches = [mpatches.Patch(color=c, label=l)
               for c,l in [('#d32f2f','Temporale'),('#388e3c','Topografico'),('#616161','Coordinate')]]
    ax.legend(handles=patches, fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'xai_barplot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvato: xai_barplot.png')

    print(f'\nBaseline MAE medio: {np.nanmean(mae_baseline):.2f} mm')
    print(f'{"Rank":>4}  {"Channel":<18s} {"Group":<10s}  {"mean":>8s}  {"std":>6s}')
    for r,i in enumerate(order,1):
        print(f'  {r:>2d}  {channel_names[i]:<18s} {channel_group[i]:<10s}  '
              f'{imp_mean[i]:>+8.3f}  {imp_std[i]:>6.3f}')

In [ ]:
# ── Performance per bacino sui fold LOYO ──────────────────────────────────────
# Carica U-Net (identica all'architettura di training)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0: g -= 1
        layers = [nn.Conv2d(in_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU(),
                  nn.Conv2d(out_ch,out_ch,3,padding=1), nn.GroupNorm(g,out_ch), nn.GELU()]
        if dropout > 0: layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def pad_to_mult(x, m=8):
    h,w = x.shape[-2],x.shape[-1]
    H = ((h+m-1)//m)*m; W = ((w+m-1)//m)*m
    ph = H-h; pw = W-w
    pl = pw//2; pr = pw-pl; pt = ph//2; pb = ph-pt
    return F.pad(x,(pl,pr,pt,pb),mode='reflect'),(pl,pr,pt,pb)

def unpad(x, p):
    pl,pr,pt,pb = p; H,W = x.shape[-2],x.shape[-1]
    return x[..., pt:H-pb if pb>0 else H, pl:W-pr if pr>0 else W]

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32, dropout=0.1):
        super().__init__()
        self.e1=DoubleConv(in_channels,base); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(base,base*2);      self.p2=nn.MaxPool2d(2)
        self.e3=DoubleConv(base*2,base*4);    self.p3=nn.MaxPool2d(2)
        self.bn=DoubleConv(base*4,base*8,dropout=dropout)
        self.u1=nn.ConvTranspose2d(base*8,base*4,2,stride=2); self.d1=DoubleConv(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2); self.d2=DoubleConv(base*4,base*2)
        self.u3=nn.ConvTranspose2d(base*2,base,2,stride=2);   self.d3=DoubleConv(base*2,base)
        self.out=nn.Conv2d(base,out_channels,1)
    def forward(self,x):
        x,p=pad_to_mult(x); e1=self.e1(x); e2=self.e2(self.p1(e1))
        e3=self.e3(self.p2(e2)); bn=self.bn(self.p3(e3))
        d1=self.d1(torch.cat([self.u1(bn),e3],1))
        d2=self.d2(torch.cat([self.u2(d1),e2],1))
        d3=self.d3(torch.cat([self.u3(d2),e1],1))
        return unpad(self.out(d3),p)

print('Architettura U-Net definita.')

In [ ]:
# ── Carica topo_tensor da AUX_DIR ────────────────────────────────────────────
STATIC_FILE = AUX_DIR / 'sportlis_static_extended.nc'
CANOPY_FILE = AUX_DIR / 'sportlis_canopy_extended_3km.nc'
TPI_RADII   = {'tpi_short': 5, 'tpi_long': 10}

ds_static = xr.open_dataset(STATIC_FILE)
static_dict = {v: ds_static[v].values.astype(np.float32)
               for v in ['elevation','slope','aspect_sin','aspect_cos']}
elev = static_dict['elevation']
elev_filled = np.where(np.isfinite(elev), elev, np.nanmedian(elev)).astype(np.float32)

tpi_layers = {}
for name, r in TPI_RADII.items():
    loc_mean = uniform_filter(elev_filled, size=2*r+1, mode='nearest').astype(np.float32)
    tpi_layers[name] = (elev_filled - loc_mean).astype(np.float32)

ds_canopy = xr.open_dataset(CANOPY_FILE)
canopy_fraction = ds_canopy['canopy_fraction'].values.astype(np.float32)

topo_layers_raw = {}
for v in ['elevation','slope','aspect_sin','aspect_cos']:
    arr = static_dict[v].astype(np.float32)
    if v in ('aspect_sin','aspect_cos'): arr = np.where(np.isfinite(arr), arr, 0.0)
    else: arr = np.where(np.isnan(arr), np.nanmedian(arr), arr)
    topo_layers_raw[v] = arr
for name, arr in tpi_layers.items():
    topo_layers_raw[name] = arr
topo_layers_raw['canopy_fraction'] = canopy_fraction

topo_normalized = []
for name in TOPO_VARS:
    arr = topo_layers_raw[name]
    if name in ('aspect_sin','aspect_cos'):
        topo_normalized.append(arr)
    elif name == 'canopy_fraction':
        topo_normalized.append((arr - float(np.nanmean(arr))).astype(np.float32))
    else:
        m = float(np.nanmean(arr)); s = float(np.nanstd(arr)) or 1.0
        topo_normalized.append(((arr-m)/s).astype(np.float32))

topo_tensor = np.stack(topo_normalized, axis=0).astype(np.float32)
print(f'topo_tensor: {topo_tensor.shape}')

In [ ]:
# ── Funzione inference per un anno ────────────────────────────────────────────
def load_stats(test_year):
    """Carica stats normalizzazione dalla cache."""
    sc = np.load(str(OUTPUT_DIR / f'norm_stats_test{test_year}.npz'), allow_pickle=True)
    mean_arr = sc['mean_arr']; std_arr = sc['std_arr']
    mean_d = {v: float(mean_arr[i]) for i,v in enumerate(INPUT_VARS)}
    std_d  = {v: float(std_arr[i])  for i,v in enumerate(INPUT_VARS)}
    return mean_d, std_d

def open_zarr_robust(path):
    p = str(path)
    for kw in [dict(consolidated=True), dict(consolidated=False),
               dict(consolidated=True,zarr_format=3), dict(consolidated=False,zarr_format=3)]:
        try: return xr.open_zarr(p, chunks={}, **kw)
        except: continue
    return xr.open_zarr(p, chunks={})

def zarr_path(year):
    return ZARR_DIR / f'sportlis_narr_pp_{year}.zarr'

def get_closest_fold(year):
    """Trova il fold più vicino il cui test year è più lontano dall'anno richiesto."""
    test_years = [f['test'] for f in LOYO_FOLDS]
    # Usa il fold che NON ha questo anno come test (preferibilmente)
    # Se l'anno è un test year, usa quel fold direttamente
    if year in test_years:
        return year
    # Altrimenti usa il fold con test year più vicino
    return min(test_years, key=lambda ty: abs(ty - year))

def predict_year(year, fold_test_year=None, batch_size=4):
    """Esegue inference per un anno intero. Restituisce (T, H, W) in mm."""
    if fold_test_year is None:
        fold_test_year = get_closest_fold(year)
    
    ckpt_path = OUTPUT_DIR / CHECKPOINT_TEMPLATE.format(test=fold_test_year)
    if not ckpt_path.exists():
        print(f'  Checkpoint non trovato: {ckpt_path}'); return None, None
    
    mean_d, std_d = load_stats(fold_test_year)
    
    # Carica zarr
    ds = open_zarr_robust(zarr_path(year))
    times = pd.to_datetime(ds['time'].values)
    mask_snowy = ds['swe_mask'].values.astype(np.float32)  # (T, H, W)
    
    # Costruisce input features
    lat2d, lon2d = np.meshgrid(lat_1d, lon_1d, indexing='ij')
    lat_norm = ((lat2d - lat2d.mean()) / (lat2d.std() or 1.0)).astype(np.float32)
    lon_norm = ((lon2d - lon2d.mean()) / (lon2d.std() or 1.0)).astype(np.float32)
    
    feats = []
    for v in INPUT_VARS:
        arr = ds[v].values.astype(np.float32)  # (T, H, W)
        arr = (arr - mean_d[v]) / std_d[v]
        feats.append(arr)
    feats.append(np.broadcast_to(lat_norm[None], (ds.sizes['time'], EXT_H, EXT_W)).copy())
    feats.append(np.broadcast_to(lon_norm[None], (ds.sizes['time'], EXT_H, EXT_W)).copy())
    X = np.stack(feats, axis=1).astype(np.float32)  # (T, 9, H, W)
    topo_t = np.broadcast_to(topo_tensor[None], (ds.sizes['time'], 7, EXT_H, EXT_W)).copy()
    X = np.concatenate([X, topo_t], axis=1)  # (T, 16, H, W)
    
    # Modello
    model = UNet(N_IN_CHANNELS, dropout=DROPOUT_P).to(device)
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    model.eval()
    
    T = X.shape[0]
    preds = np.zeros((T, EXT_H, EXT_W), dtype=np.float32)
    with torch.no_grad():
        for t0 in range(0, T, batch_size):
            t1 = min(t0+batch_size, T)
            xb = torch.from_numpy(X[t0:t1]).to(device)
            pb = model(xb).cpu().numpy()[:,0]
            preds[t0:t1] = np.expm1(np.clip(pb, 0, 10))
    
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    
    # SWE osservato
    swe_obs = ds[TARGET_VAR].values.astype(np.float32)
    swe_obs = np.expm1(np.clip(swe_obs, 0, 10))
    
    print(f'  {year}: T={T}  pred_mean={preds[mask_snowy>0].mean():.1f} mm  '
          f'obs_mean={swe_obs[mask_snowy>0].mean():.1f} mm')
    return preds, swe_obs, times, mask_snowy

print('Funzioni inference definite.')

In [ ]:
# ── Time series per bacino — anni di test (out-of-sample) ─────────────────────
# Per ogni fold usa il modello corrispondente (vera previsione OOS)

TEST_YEARS = [f['test'] for f in LOYO_FOLDS]  # [1985,1990,1995,2000,2005,2010,2015,2020]

basin_ts = {name: {'year':[], 'pred_mean':[], 'obs_mean':[], 'mae':[], 'bias':[]}
            for name in BASINS}

for ty in TEST_YEARS:
    print(f'\nInference anno {ty} (fold test={ty})...')
    result = predict_year(ty, fold_test_year=ty)
    if result[0] is None: continue
    preds, swe_obs, times, mask_snowy = result
    
    for name, info in BASINS.items():
        bmask = basin_masks[name]  # (H, W)
        valid = (mask_snowy > 0) & bmask[None, :, :]  # (T, H, W)
        if valid.sum() == 0:
            basin_ts[name]['year'].append(ty)
            for k in ['pred_mean','obs_mean','mae','bias']: basin_ts[name][k].append(np.nan)
            continue
        pm = preds[valid].mean()
        om = swe_obs[valid].mean()
        mae = np.abs(preds[valid] - swe_obs[valid]).mean()
        bias = (preds[valid] - swe_obs[valid]).mean()
        basin_ts[name]['year'].append(ty)
        basin_ts[name]['pred_mean'].append(pm)
        basin_ts[name]['obs_mean'].append(om)
        basin_ts[name]['mae'].append(mae)
        basin_ts[name]['bias'].append(bias)

print('\nDone.')

In [ ]:
# ── Plot: MAE per bacino nel tempo ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.flatten()

for ax, (name, info) in zip(axes, BASINS.items()):
    ts = basin_ts[name]
    ax.bar(ts['year'], ts['mae'], color=info['color'], alpha=0.8, width=3)
    ax.set_title(info['label'], fontsize=10)
    ax.set_xlabel('Anno test'); ax.set_ylabel('MAE (mm)')
    ax.set_xticks(ts['year']); ax.set_xticklabels(ts['year'], rotation=45, fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('MAE SWE per bacino — fold LOYO out-of-sample', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'mae_per_basin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: mae_per_basin.png')

In [ ]:
# ── Plot: SWE predetto vs osservato per bacino (anni test) ────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (name, info) in zip(axes, BASINS.items()):
    ts = basin_ts[name]
    ax.plot(ts['year'], ts['obs_mean'],  'o-', color='k', label='SPORTLIS', lw=2)
    ax.plot(ts['year'], ts['pred_mean'], 's--', color=info['color'], label='U-Net', lw=2)
    ax.set_title(info['label'], fontsize=10)
    ax.set_xlabel('Anno test'); ax.set_ylabel('SWE medio (mm)')
    ax.legend(fontsize=8)
    ax.set_xticks(ts['year']); ax.set_xticklabels(ts['year'], rotation=45, fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('SWE medio per bacino: U-Net vs SPORTLIS (fold LOYO)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'swe_pred_vs_obs_basin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: swe_pred_vs_obs_basin.png')

In [ ]:
# ── Proiezione 2025–2026 ──────────────────────────────────────────────────────
# Usa il modello dell'ultimo fold (test=2020) per proiettare 2025 e 2026
# Richiede che i zarr 2025/2026 siano stati costruiti (NARR fino a 2024 scaricato)

PROJ_YEARS = [2025, 2026]
proj_available = [y for y in PROJ_YEARS if zarr_path(y).exists()]

if not proj_available:
    print('Zarr 2025/2026 non disponibili — esegui prima il download NARR e build zarr per questi anni.')
    print('NARR è disponibile fino a circa fine 2024. Verifica la disponibilità dei dati 2025.')
else:
    proj_results = {}
    for y in proj_available:
        print(f'\nProiezione {y}...')
        result = predict_year(y, fold_test_year=2020)  # usa modello più recente
        if result[0] is not None:
            proj_results[y] = result
    
    if proj_results:
        fig, axes = plt.subplots(1, len(proj_results), figsize=(8*len(proj_results), 5))
        if len(proj_results) == 1: axes = [axes]
        for ax, (y, (preds, _, times, mask)) in zip(axes, proj_results.items()):
            swe_domain = np.where(mask>0, preds, np.nan).mean(axis=(1,2))
            ax.plot(times, swe_domain, color='steelblue', lw=1.5)
            ax.set_title(f'SWE medio dominio — {y} (proiezione)', fontsize=11)
            ax.set_ylabel('SWE (mm)'); ax.set_xlabel('Data')
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'projection_2025_2026.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Salvato: projection_2025_2026.png')